In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import optuna
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

In [9]:
# Load your data
data = pd.read_csv('QGIS_df.csv')

In [11]:
house_df = data[data['Category'] == 'house']
condo_df = data[data['Category'] == 'condo']
apt_df = data[data['Category'] == 'apartment']
com_df = data[data['Category'] == 'commercial']
land_df = data[data['Category'] == 'land']
RR_df = pd.concat([house_df, condo_df, apt_df], ignore_index=True)
CR_df = pd.concat([com_df, land_df], ignore_index=True)

In [17]:
def objective(trial):
    # Hyperparameter ranges as specified
    params = {
        'n_estimators': trial.suggest_categorical('n_estimators', [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120, 130, 140, 150, 160, 170, 180, 190, 200]),
        'max_depth': trial.suggest_int('max_depth', 1, 20),
        'max_features': trial.suggest_int('max_features', 1, 9),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'min_samples_split': trial.suggest_int('min_samples_split', 1, 10)
    }
    
    # Create Random Forest Regressor with current hyperparameters
    rf_regressor = RandomForestRegressor(
        n_estimators=params['n_estimators'],
        max_depth=params['max_depth'],
        max_features=params['max_features'],
        min_samples_leaf=params['min_samples_leaf'],
        min_samples_split=params['min_samples_split'],
        criterion='squared_error',
        bootstrap=True
    )
    
    # Perform cross-validation
    scores = cross_val_score(rf_regressor, X, y, cv=5, scoring='r2')
    return np.mean(scores)

In [19]:
def optimize_random_forest(X, y):
    # Create a study object and optimize the objective function
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=100)
    
    # Print best parameters and score
    print("Best trial:")
    trial = study.best_trial
    print(f"  Value: {trial.value}")
    print("  Params: ")
    for key, value in trial.params.items():
        print(f"    {key}: {value}")
    
    return study.best_params

In [21]:
def get_feature_importance(model, feature_names):
    #Prints and plots feature importance from a trained RandomForestRegressor model.
    importance = model.feature_importances_
    feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importance})
    
    # Sort by importance
    feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

    # Print feature importance
    print("\nFeature Importance:")
    print(feature_importance_df)

    # Plot feature importance
    plt.figure(figsize=(10, 5))
    plt.barh(feature_importance_df['Feature'], feature_importance_df['Importance'], color='skyblue')
    plt.xlabel("Feature Importance")
    plt.ylabel("Feature")
    plt.title("Feature Importance of Random Forest")
    plt.gca().invert_yaxis()  # Flip to show highest at the top
    plt.show()

In [23]:
# Assuming df is already loaded and preprocessed
y = house_df['Price']
X = house_df[["LRTHubDist", 'Num_Bedrooms', 'Num_Bathrooms', 'Floor_Area', 'Land_Area', 
        'BusCount', 'SchoolCount', 'MallCount', 'HospitalCount']]

# Optimize and get best parameters
best_params = optimize_random_forest(X, y)

# Train final model with best parameters
final_model = RandomForestRegressor(**best_params, criterion='squared_error', bootstrap=True, random_state=42)
final_model.fit(X, y)

# Get feature importance
get_feature_importance(final_model, X.columns)

[I 2025-02-03 19:50:33,447] A new study created in memory with name: no-name-a9fd2a7d-392f-4122-815d-20fbb6081db6
[I 2025-02-03 19:50:46,102] Trial 0 finished with value: -0.1877217470241262 and parameters: {'n_estimators': 100, 'max_depth': 11, 'max_features': 5, 'min_samples_leaf': 10, 'min_samples_split': 6}. Best is trial 0 with value: -0.1877217470241262.
[W 2025-02-03 19:50:46,119] Trial 1 failed with parameters: {'n_estimators': 140, 'max_depth': 13, 'max_features': 6, 'min_samples_leaf': 7, 'min_samples_split': 1} because of the following error: ValueError('\nAll the 5 fits failed.\nIt is very likely that your model is misconfigured.\nYou can try to debug the error by setting error_score=\'raise\'.\n\nBelow are more details about the failures:\n--------------------------------------------------------------------------------\n5 fits failed with the following error:\nTraceback (most recent call last):\n  File "C:\\Users\\Dianne Yumol\\anaconda3\\Lib\\site-packages\\sklearn\\model

ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 895, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\base.py", line 1467, in wrapper
    estimator._validate_params()
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\base.py", line 666, in _validate_params
    validate_parameter_constraints(
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\utils\_param_validation.py", line 95, in validate_parameter_constraints
    raise InvalidParameterError(
sklearn.utils._param_validat

In [25]:
# Assuming df is already loaded and preprocessed
y = condo_df['Price']
X = condo_df[["LRTHubDist", 'Num_Bedrooms', 'Num_Bathrooms', 'Floor_Area', 'Land_Area', 
        'BusCount', 'SchoolCount', 'MallCount', 'HospitalCount']]

# Optimize and get best parameters
best_params = optimize_random_forest(X, y)

# Train final model with best parameters
final_model = RandomForestRegressor(**best_params, criterion='squared_error', bootstrap=True, random_state=42)
final_model.fit(X, y)

# Get feature importance
get_feature_importance(final_model, X.columns)

[I 2025-02-03 20:28:33,685] A new study created in memory with name: no-name-3f666b14-2d8d-4583-95ed-4a63419db6fd
[I 2025-02-03 20:28:36,459] Trial 0 finished with value: 0.367504266396316 and parameters: {'n_estimators': 70, 'max_depth': 9, 'max_features': 2, 'min_samples_leaf': 4, 'min_samples_split': 6}. Best is trial 0 with value: 0.367504266396316.
[I 2025-02-03 20:28:40,150] Trial 1 finished with value: 0.24237588134498048 and parameters: {'n_estimators': 190, 'max_depth': 1, 'max_features': 7, 'min_samples_leaf': 3, 'min_samples_split': 6}. Best is trial 0 with value: 0.367504266396316.
[I 2025-02-03 20:28:48,978] Trial 2 finished with value: 0.2694894752058204 and parameters: {'n_estimators': 50, 'max_depth': 14, 'max_features': 9, 'min_samples_leaf': 3, 'min_samples_split': 6}. Best is trial 0 with value: 0.367504266396316.
[I 2025-02-03 20:28:49,535] Trial 3 finished with value: 0.19192736928052254 and parameters: {'n_estimators': 40, 'max_depth': 1, 'max_features': 3, 'min_s

ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 895, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\base.py", line 1467, in wrapper
    estimator._validate_params()
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\base.py", line 666, in _validate_params
    validate_parameter_constraints(
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\utils\_param_validation.py", line 95, in validate_parameter_constraints
    raise InvalidParameterError(
sklearn.utils._param_validat

In [27]:
# Assuming df is already loaded and preprocessed
y = apt_df['Price']
X = apt_df[["LRTHubDist", 'Num_Bedrooms', 'Num_Bathrooms', 'Floor_Area', 'Land_Area', 
        'BusCount', 'SchoolCount', 'MallCount', 'HospitalCount']]

# Optimize and get best parameters
best_params = optimize_random_forest(X, y)

# Train final model with best parameters
final_model = RandomForestRegressor(**best_params, criterion='squared_error', bootstrap=True, random_state=42)
final_model.fit(X, y)

# Get feature importance
get_feature_importance(final_model, X.columns)

[I 2025-02-03 20:30:04,120] A new study created in memory with name: no-name-2d8a68d9-2a5f-47a7-8f67-30b77f800723
[I 2025-02-03 20:30:04,423] Trial 0 finished with value: 0.3095963246646894 and parameters: {'n_estimators': 40, 'max_depth': 1, 'max_features': 4, 'min_samples_leaf': 3, 'min_samples_split': 2}. Best is trial 0 with value: 0.3095963246646894.
[I 2025-02-03 20:30:05,565] Trial 1 finished with value: 0.42107728400786676 and parameters: {'n_estimators': 190, 'max_depth': 18, 'max_features': 4, 'min_samples_leaf': 6, 'min_samples_split': 10}. Best is trial 1 with value: 0.42107728400786676.
[I 2025-02-03 20:30:06,359] Trial 2 finished with value: 0.2726970629515328 and parameters: {'n_estimators': 120, 'max_depth': 17, 'max_features': 7, 'min_samples_leaf': 10, 'min_samples_split': 3}. Best is trial 1 with value: 0.42107728400786676.
[I 2025-02-03 20:30:06,805] Trial 3 finished with value: 0.3902121198891145 and parameters: {'n_estimators': 50, 'max_depth': 10, 'max_features':

ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 895, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\base.py", line 1467, in wrapper
    estimator._validate_params()
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\base.py", line 666, in _validate_params
    validate_parameter_constraints(
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\utils\_param_validation.py", line 95, in validate_parameter_constraints
    raise InvalidParameterError(
sklearn.utils._param_validat

In [31]:
# Assuming df is already loaded and preprocessed
y = com_df['Price']
X = com_df[["LRTHubDist", 'Num_Bedrooms', 'Num_Bathrooms', 'Floor_Area', 'Land_Area', 
        'BusCount', 'SchoolCount', 'MallCount', 'HospitalCount']]

# Optimize and get best parameters
best_params = optimize_random_forest(X, y)

# Train final model with best parameters
final_model = RandomForestRegressor(**best_params, criterion='squared_error', bootstrap=True, random_state=42)
final_model.fit(X, y)

# Get feature importance
get_feature_importance(final_model, X.columns)

[I 2025-02-03 21:17:09,511] A new study created in memory with name: no-name-7d4fe011-8ddd-4282-b102-64389a09683f
[I 2025-02-03 21:17:13,807] Trial 0 finished with value: 0.4139536808226594 and parameters: {'n_estimators': 160, 'max_depth': 19, 'max_features': 7, 'min_samples_leaf': 4, 'min_samples_split': 7}. Best is trial 0 with value: 0.4139536808226594.
[I 2025-02-03 21:17:14,981] Trial 1 finished with value: 0.376392673922475 and parameters: {'n_estimators': 150, 'max_depth': 5, 'max_features': 2, 'min_samples_leaf': 6, 'min_samples_split': 10}. Best is trial 0 with value: 0.4139536808226594.
[I 2025-02-03 21:17:16,973] Trial 2 finished with value: 0.4260666404178261 and parameters: {'n_estimators': 90, 'max_depth': 18, 'max_features': 6, 'min_samples_leaf': 5, 'min_samples_split': 3}. Best is trial 2 with value: 0.4260666404178261.
[I 2025-02-03 21:17:18,301] Trial 3 finished with value: 0.32151280560515383 and parameters: {'n_estimators': 200, 'max_depth': 3, 'max_features': 2, 

ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 895, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\base.py", line 1467, in wrapper
    estimator._validate_params()
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\base.py", line 666, in _validate_params
    validate_parameter_constraints(
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\utils\_param_validation.py", line 95, in validate_parameter_constraints
    raise InvalidParameterError(
sklearn.utils._param_validat

In [33]:
# Assuming df is already loaded and preprocessed
y = land_df['Price']
X = land_df[["LRTHubDist", 'Num_Bedrooms', 'Num_Bathrooms', 'Floor_Area', 'Land_Area', 
        'BusCount', 'SchoolCount', 'MallCount', 'HospitalCount']]

# Optimize and get best parameters
best_params = optimize_random_forest(X, y)

# Train final model with best parameters
final_model = RandomForestRegressor(**best_params, criterion='squared_error', bootstrap=True, random_state=42)
final_model.fit(X, y)

# Get feature importance
get_feature_importance(final_model, X.columns)

[I 2025-02-03 21:17:37,283] A new study created in memory with name: no-name-46af8242-e258-44ff-bfdc-80727d75c457
[I 2025-02-03 21:17:38,111] Trial 0 finished with value: -4.349713262671183 and parameters: {'n_estimators': 30, 'max_depth': 18, 'max_features': 1, 'min_samples_leaf': 1, 'min_samples_split': 4}. Best is trial 0 with value: -4.349713262671183.
[W 2025-02-03 21:17:38,126] Trial 1 failed with parameters: {'n_estimators': 150, 'max_depth': 12, 'max_features': 1, 'min_samples_leaf': 9, 'min_samples_split': 1} because of the following error: ValueError('\nAll the 5 fits failed.\nIt is very likely that your model is misconfigured.\nYou can try to debug the error by setting error_score=\'raise\'.\n\nBelow are more details about the failures:\n--------------------------------------------------------------------------------\n5 fits failed with the following error:\nTraceback (most recent call last):\n  File "C:\\Users\\Dianne Yumol\\anaconda3\\Lib\\site-packages\\sklearn\\model_sel

ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 895, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\base.py", line 1467, in wrapper
    estimator._validate_params()
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\base.py", line 666, in _validate_params
    validate_parameter_constraints(
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\utils\_param_validation.py", line 95, in validate_parameter_constraints
    raise InvalidParameterError(
sklearn.utils._param_validat

In [35]:
# Assuming df is already loaded and preprocessed
y = RR_df['Price']
X = RR_df[["LRTHubDist", 'Num_Bedrooms', 'Num_Bathrooms', 'Floor_Area', 'Land_Area', 
        'BusCount', 'SchoolCount', 'MallCount', 'HospitalCount']]

# Optimize and get best parameters
best_params = optimize_random_forest(X, y)

# Train final model with best parameters
final_model = RandomForestRegressor(**best_params, criterion='squared_error', bootstrap=True, random_state=42)
final_model.fit(X, y)

# Get feature importance
get_feature_importance(final_model, X.columns)

[I 2025-02-03 21:42:36,262] A new study created in memory with name: no-name-c7d36f9a-bdcb-469a-ab31-6a549f0e6565
[I 2025-02-03 21:42:49,819] Trial 0 finished with value: 0.24903947511464436 and parameters: {'n_estimators': 130, 'max_depth': 5, 'max_features': 1, 'min_samples_leaf': 9, 'min_samples_split': 9}. Best is trial 0 with value: 0.24903947511464436.
[I 2025-02-03 21:45:21,165] Trial 1 finished with value: 0.27811829175944014 and parameters: {'n_estimators': 120, 'max_depth': 18, 'max_features': 8, 'min_samples_leaf': 9, 'min_samples_split': 4}. Best is trial 1 with value: 0.27811829175944014.
[I 2025-02-03 21:45:31,681] Trial 2 finished with value: 0.12481721450187884 and parameters: {'n_estimators': 120, 'max_depth': 1, 'max_features': 5, 'min_samples_leaf': 1, 'min_samples_split': 7}. Best is trial 1 with value: 0.27811829175944014.
[I 2025-02-03 21:45:36,309] Trial 3 finished with value: 0.25700773913383934 and parameters: {'n_estimators': 10, 'max_depth': 5, 'max_features'

ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 895, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\base.py", line 1467, in wrapper
    estimator._validate_params()
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\base.py", line 666, in _validate_params
    validate_parameter_constraints(
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\utils\_param_validation.py", line 95, in validate_parameter_constraints
    raise InvalidParameterError(
sklearn.utils._param_validat

In [37]:
# Assuming df is already loaded and preprocessed
y = CR_df['Price']
X = CR_df[["LRTHubDist", 'Num_Bedrooms', 'Num_Bathrooms', 'Floor_Area', 'Land_Area', 
        'BusCount', 'SchoolCount', 'MallCount', 'HospitalCount']]

# Optimize and get best parameters
best_params = optimize_random_forest(X, y)

# Train final model with best parameters
final_model = RandomForestRegressor(**best_params, criterion='squared_error', bootstrap=True, random_state=42)
final_model.fit(X, y)

# Get feature importance
get_feature_importance(final_model, X.columns)

[I 2025-02-03 22:05:47,512] A new study created in memory with name: no-name-4e572ebb-79af-4366-b172-f929a4f2f47a
[W 2025-02-03 22:05:47,547] Trial 0 failed with parameters: {'n_estimators': 60, 'max_depth': 18, 'max_features': 6, 'min_samples_leaf': 1, 'min_samples_split': 1} because of the following error: ValueError('\nAll the 5 fits failed.\nIt is very likely that your model is misconfigured.\nYou can try to debug the error by setting error_score=\'raise\'.\n\nBelow are more details about the failures:\n--------------------------------------------------------------------------------\n5 fits failed with the following error:\nTraceback (most recent call last):\n  File "C:\\Users\\Dianne Yumol\\anaconda3\\Lib\\site-packages\\sklearn\\model_selection\\_validation.py", line 895, in _fit_and_score\n    estimator.fit(X_train, y_train, **fit_params)\n  File "C:\\Users\\Dianne Yumol\\anaconda3\\Lib\\site-packages\\sklearn\\base.py", line 1467, in wrapper\n    estimator._validate_params()\n 

ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 895, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\base.py", line 1467, in wrapper
    estimator._validate_params()
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\base.py", line 666, in _validate_params
    validate_parameter_constraints(
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\utils\_param_validation.py", line 95, in validate_parameter_constraints
    raise InvalidParameterError(
sklearn.utils._param_validat

In [39]:
# Assuming df is already loaded and preprocessed
y = house_df['Price']
X = house_df[["LRTHubDist", 'Num_Bedrooms', 'Num_Bathrooms', 'Floor_Area', 'Land_Area', 
        'BusHubDist', 'SchoolHubDist', 'MallHubDist', 'HospitalHubDist']]

# Optimize and get best parameters
best_params = optimize_random_forest(X, y)

# Train final model with best parameters
final_model = RandomForestRegressor(**best_params, criterion='squared_error', bootstrap=True, random_state=42)
final_model.fit(X, y)

# Get feature importance
get_feature_importance(final_model, X.columns)

[I 2025-02-03 22:06:31,500] A new study created in memory with name: no-name-6b9c126e-0547-4f30-9c28-f78379e31d61
[I 2025-02-03 22:06:40,258] Trial 0 finished with value: -1.2843039552799804 and parameters: {'n_estimators': 130, 'max_depth': 2, 'max_features': 2, 'min_samples_leaf': 6, 'min_samples_split': 5}. Best is trial 0 with value: -1.2843039552799804.
[I 2025-02-03 22:07:48,492] Trial 1 finished with value: -0.06280089131705632 and parameters: {'n_estimators': 190, 'max_depth': 14, 'max_features': 3, 'min_samples_leaf': 4, 'min_samples_split': 2}. Best is trial 1 with value: -0.06280089131705632.
[W 2025-02-03 22:07:48,550] Trial 2 failed with parameters: {'n_estimators': 100, 'max_depth': 4, 'max_features': 8, 'min_samples_leaf': 5, 'min_samples_split': 1} because of the following error: ValueError('\nAll the 5 fits failed.\nIt is very likely that your model is misconfigured.\nYou can try to debug the error by setting error_score=\'raise\'.\n\nBelow are more details about the f

ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 895, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\base.py", line 1467, in wrapper
    estimator._validate_params()
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\base.py", line 666, in _validate_params
    validate_parameter_constraints(
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\utils\_param_validation.py", line 95, in validate_parameter_constraints
    raise InvalidParameterError(
sklearn.utils._param_validat

In [41]:
# Assuming df is already loaded and preprocessed
y = condo_df['Price']
X = condo_df[["LRTHubDist", 'Num_Bedrooms', 'Num_Bathrooms', 'Floor_Area', 'Land_Area', 
        'BusHubDist', 'SchoolHubDist', 'MallHubDist', 'HospitalHubDist']]

# Optimize and get best parameters
best_params = optimize_random_forest(X, y)

# Train final model with best parameters
final_model = RandomForestRegressor(**best_params, criterion='squared_error', bootstrap=True, random_state=42)
final_model.fit(X, y)

# Get feature importance
get_feature_importance(final_model, X.columns)

[I 2025-02-03 22:26:31,186] A new study created in memory with name: no-name-804e53df-cb89-4410-bcd2-e2c89c68b0c7
[I 2025-02-03 22:27:03,146] Trial 0 finished with value: 0.4136895451185965 and parameters: {'n_estimators': 120, 'max_depth': 12, 'max_features': 4, 'min_samples_leaf': 4, 'min_samples_split': 8}. Best is trial 0 with value: 0.4136895451185965.
[I 2025-02-03 22:27:11,215] Trial 1 finished with value: 0.40316684692765953 and parameters: {'n_estimators': 20, 'max_depth': 19, 'max_features': 5, 'min_samples_leaf': 8, 'min_samples_split': 2}. Best is trial 0 with value: 0.4136895451185965.
[I 2025-02-03 22:27:20,685] Trial 2 finished with value: 0.39065530860428405 and parameters: {'n_estimators': 40, 'max_depth': 16, 'max_features': 3, 'min_samples_leaf': 6, 'min_samples_split': 10}. Best is trial 0 with value: 0.4136895451185965.
[I 2025-02-03 22:27:51,789] Trial 3 finished with value: 0.3396499670748368 and parameters: {'n_estimators': 70, 'max_depth': 8, 'max_features': 9,

ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 895, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\base.py", line 1467, in wrapper
    estimator._validate_params()
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\base.py", line 666, in _validate_params
    validate_parameter_constraints(
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\utils\_param_validation.py", line 95, in validate_parameter_constraints
    raise InvalidParameterError(
sklearn.utils._param_validat

In [43]:
# Assuming df is already loaded and preprocessed
y = apt_df['Price']
X = apt_df[["LRTHubDist", 'Num_Bedrooms', 'Num_Bathrooms', 'Floor_Area', 'Land_Area', 
        'BusHubDist', 'SchoolHubDist', 'MallHubDist', 'HospitalHubDist']]

# Optimize and get best parameters
best_params = optimize_random_forest(X, y)

# Train final model with best parameters
final_model = RandomForestRegressor(**best_params, criterion='squared_error', bootstrap=True, random_state=42)
final_model.fit(X, y)

# Get feature importance
get_feature_importance(final_model, X.columns)

[I 2025-02-03 22:28:21,491] A new study created in memory with name: no-name-1bb5040e-842d-4437-b54f-4b1e94d18b7b
[I 2025-02-03 22:28:24,322] Trial 0 finished with value: 0.35624083383051386 and parameters: {'n_estimators': 150, 'max_depth': 8, 'max_features': 4, 'min_samples_leaf': 10, 'min_samples_split': 4}. Best is trial 0 with value: 0.35624083383051386.
[I 2025-02-03 22:28:26,331] Trial 1 finished with value: 0.037204086602074506 and parameters: {'n_estimators': 70, 'max_depth': 14, 'max_features': 7, 'min_samples_leaf': 1, 'min_samples_split': 3}. Best is trial 0 with value: 0.35624083383051386.
[I 2025-02-03 22:28:29,061] Trial 2 finished with value: 0.4001809062335093 and parameters: {'n_estimators': 130, 'max_depth': 10, 'max_features': 5, 'min_samples_leaf': 6, 'min_samples_split': 6}. Best is trial 2 with value: 0.4001809062335093.
[I 2025-02-03 22:28:32,431] Trial 3 finished with value: 0.22226438330630788 and parameters: {'n_estimators': 180, 'max_depth': 10, 'max_feature

ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 895, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\base.py", line 1467, in wrapper
    estimator._validate_params()
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\base.py", line 666, in _validate_params
    validate_parameter_constraints(
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\utils\_param_validation.py", line 95, in validate_parameter_constraints
    raise InvalidParameterError(
sklearn.utils._param_validat

In [45]:
# Assuming df is already loaded and preprocessed
y = com_df['Price']
X = com_df[["LRTHubDist", 'Num_Bedrooms', 'Num_Bathrooms', 'Floor_Area', 'Land_Area', 
        'BusHubDist', 'SchoolHubDist', 'MallHubDist', 'HospitalHubDist']]

# Optimize and get best parameters
best_params = optimize_random_forest(X, y)

# Train final model with best parameters
final_model = RandomForestRegressor(**best_params, criterion='squared_error', bootstrap=True, random_state=42)
final_model.fit(X, y)

# Get feature importance
get_feature_importance(final_model, X.columns)

[I 2025-02-03 22:30:11,671] A new study created in memory with name: no-name-585e7f49-6fa0-4097-b190-10c81ff47313
[I 2025-02-03 22:30:12,207] Trial 0 finished with value: 0.3840828434470317 and parameters: {'n_estimators': 10, 'max_depth': 20, 'max_features': 3, 'min_samples_leaf': 9, 'min_samples_split': 10}. Best is trial 0 with value: 0.3840828434470317.
[I 2025-02-03 22:30:23,053] Trial 1 finished with value: 0.4275522716942416 and parameters: {'n_estimators': 160, 'max_depth': 18, 'max_features': 4, 'min_samples_leaf': 4, 'min_samples_split': 5}. Best is trial 1 with value: 0.4275522716942416.
[I 2025-02-03 22:30:37,374] Trial 2 finished with value: 0.38431399283112705 and parameters: {'n_estimators': 160, 'max_depth': 12, 'max_features': 7, 'min_samples_leaf': 7, 'min_samples_split': 7}. Best is trial 1 with value: 0.4275522716942416.
[I 2025-02-03 22:30:38,209] Trial 3 finished with value: 0.3754105509212334 and parameters: {'n_estimators': 10, 'max_depth': 10, 'max_features': 6

ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 895, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\base.py", line 1467, in wrapper
    estimator._validate_params()
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\base.py", line 666, in _validate_params
    validate_parameter_constraints(
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\utils\_param_validation.py", line 95, in validate_parameter_constraints
    raise InvalidParameterError(
sklearn.utils._param_validat

In [47]:
# Assuming df is already loaded and preprocessed
y = land_df['Price']
X = land_df[["LRTHubDist", 'Num_Bedrooms', 'Num_Bathrooms', 'Floor_Area', 'Land_Area', 
        'BusHubDist', 'SchoolHubDist', 'MallHubDist', 'HospitalHubDist']]

# Optimize and get best parameters
best_params = optimize_random_forest(X, y)

# Train final model with best parameters
final_model = RandomForestRegressor(**best_params, criterion='squared_error', bootstrap=True, random_state=42)
final_model.fit(X, y)

# Get feature importance
get_feature_importance(final_model, X.columns)

[I 2025-02-03 22:32:27,563] A new study created in memory with name: no-name-e2f8ced2-6612-4d0c-b7f4-66a09976d768
[I 2025-02-03 22:32:30,139] Trial 0 finished with value: -3.496028629366299 and parameters: {'n_estimators': 50, 'max_depth': 4, 'max_features': 3, 'min_samples_leaf': 5, 'min_samples_split': 8}. Best is trial 0 with value: -3.496028629366299.
[I 2025-02-03 22:32:46,451] Trial 1 finished with value: -12.406423411978315 and parameters: {'n_estimators': 200, 'max_depth': 5, 'max_features': 6, 'min_samples_leaf': 10, 'min_samples_split': 3}. Best is trial 0 with value: -3.496028629366299.
[I 2025-02-03 22:32:56,741] Trial 2 finished with value: -4.945233741663134 and parameters: {'n_estimators': 100, 'max_depth': 9, 'max_features': 5, 'min_samples_leaf': 10, 'min_samples_split': 9}. Best is trial 0 with value: -3.496028629366299.
[I 2025-02-03 22:33:08,395] Trial 3 finished with value: -10.59513404620206 and parameters: {'n_estimators': 150, 'max_depth': 7, 'max_features': 4, 

ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 895, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\base.py", line 1467, in wrapper
    estimator._validate_params()
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\base.py", line 666, in _validate_params
    validate_parameter_constraints(
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\utils\_param_validation.py", line 95, in validate_parameter_constraints
    raise InvalidParameterError(
sklearn.utils._param_validat

In [49]:
# Assuming df is already loaded and preprocessed
y = RR_df['Price']
X = RR_df[["LRTHubDist", 'Num_Bedrooms', 'Num_Bathrooms', 'Floor_Area', 'Land_Area', 
        'BusHubDist', 'SchoolHubDist', 'MallHubDist', 'HospitalHubDist']]

# Optimize and get best parameters
best_params = optimize_random_forest(X, y)

# Train final model with best parameters
final_model = RandomForestRegressor(**best_params, criterion='squared_error', bootstrap=True, random_state=42)
final_model.fit(X, y)

# Get feature importance
get_feature_importance(final_model, X.columns)

[I 2025-02-03 22:41:19,728] A new study created in memory with name: no-name-55390329-d254-4e23-820a-bcfa93664feb
[I 2025-02-03 22:42:10,548] Trial 0 finished with value: 0.29931756011994665 and parameters: {'n_estimators': 60, 'max_depth': 19, 'max_features': 4, 'min_samples_leaf': 10, 'min_samples_split': 9}. Best is trial 0 with value: 0.29931756011994665.
[I 2025-02-03 22:42:30,274] Trial 1 finished with value: 0.2949061291746224 and parameters: {'n_estimators': 80, 'max_depth': 11, 'max_features': 1, 'min_samples_leaf': 6, 'min_samples_split': 6}. Best is trial 0 with value: 0.29931756011994665.
[I 2025-02-03 22:44:27,308] Trial 2 finished with value: 0.03208706826060108 and parameters: {'n_estimators': 70, 'max_depth': 15, 'max_features': 8, 'min_samples_leaf': 3, 'min_samples_split': 10}. Best is trial 0 with value: 0.29931756011994665.
[I 2025-02-03 22:44:46,823] Trial 3 finished with value: 0.20814972198376705 and parameters: {'n_estimators': 170, 'max_depth': 2, 'max_features

ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 895, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\base.py", line 1467, in wrapper
    estimator._validate_params()
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\base.py", line 666, in _validate_params
    validate_parameter_constraints(
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\utils\_param_validation.py", line 95, in validate_parameter_constraints
    raise InvalidParameterError(
sklearn.utils._param_validat

In [51]:
# Assuming df is already loaded and preprocessed
y = CR_df['Price']
X = CR_df[["LRTHubDist", 'Num_Bedrooms', 'Num_Bathrooms', 'Floor_Area', 'Land_Area', 
        'BusHubDist', 'SchoolHubDist', 'MallHubDist', 'HospitalHubDist']]

# Optimize and get best parameters
best_params = optimize_random_forest(X, y)

# Train final model with best parameters
final_model = RandomForestRegressor(**best_params, criterion='squared_error', bootstrap=True, random_state=42)
final_model.fit(X, y)

# Get feature importance
get_feature_importance(final_model, X.columns)

[I 2025-02-03 23:03:40,125] A new study created in memory with name: no-name-556e0efe-3f8f-49e6-b3b7-4265b9f29670
[I 2025-02-03 23:03:40,874] Trial 0 finished with value: -0.009850577924727788 and parameters: {'n_estimators': 20, 'max_depth': 3, 'max_features': 1, 'min_samples_leaf': 7, 'min_samples_split': 10}. Best is trial 0 with value: -0.009850577924727788.
[I 2025-02-03 23:03:44,048] Trial 1 finished with value: -0.2553602614291653 and parameters: {'n_estimators': 70, 'max_depth': 8, 'max_features': 1, 'min_samples_leaf': 9, 'min_samples_split': 3}. Best is trial 0 with value: -0.009850577924727788.
[I 2025-02-03 23:03:47,725] Trial 2 finished with value: -0.02618598231914131 and parameters: {'n_estimators': 130, 'max_depth': 1, 'max_features': 2, 'min_samples_leaf': 10, 'min_samples_split': 5}. Best is trial 0 with value: -0.009850577924727788.
[I 2025-02-03 23:03:50,368] Trial 3 finished with value: -5.52452580518874 and parameters: {'n_estimators': 10, 'max_depth': 18, 'max_fe

ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 895, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\base.py", line 1467, in wrapper
    estimator._validate_params()
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\base.py", line 666, in _validate_params
    validate_parameter_constraints(
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\utils\_param_validation.py", line 95, in validate_parameter_constraints
    raise InvalidParameterError(
sklearn.utils._param_validat